In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

import seaborn as sns
from helper_functions_robustness import *

## Pre-process the robustness

In [2]:
# setup attributes
sim_mode = 'p90'
action_mode = 'IU'
num_sol_dict = {'base': 46, 'IU': 628}
sim_mode_colors = {'base': '#6B8F71', 'IU': '#F2B880'}
utilities = ['Watertown', 'Dryville', 'Fallsland']
util_abbrevs = ['W', 'D', 'F']
obj_names = ['REL', 'RF', 'INPC', 'PFC', 'WCC']
objs_allutils = [obj_name + f'_{util_abbrev}' for util_abbrev in util_abbrevs for obj_name in obj_names]
objs_regional = [obj_name + '_R' for obj_name in obj_names]

In [3]:
# read in robustness 
robustness_df = pd.read_csv(f'robustness_sim_{sim_mode}_refset_{action_mode}.csv', header=0)
robustness_df_noReg = robustness_df.drop(columns=['Regional'])

# normalize robustness values to [0,1]
robustness_df_norm = (robustness_df - robustness_df.min().min()) / (robustness_df.max().max() - robustness_df.min().min())
robustness_df_norm_noReg = robustness_df_norm.drop(columns=['Regional'])

# import the high-cooperation solutions CSV 
high_coop_sols_IU = np.loadtxt(f"../objs_dvs/sol_high_coop_{action_mode}.csv").flatten().astype(int)

In [4]:
# filter out high_coop_sols that have lower perturbation than the baseline solution 
selected_baseline_idx = 29 
percent_perturbation_baseline_df = pd.read_csv(f"percent_degradation_max_allsols_base_p90.csv", header=0)  
# filter out everything that contains 'INPC' or 'PFC' since those are not relevant to the high-cooperation solutions
cols_to_drop = [col for col in percent_perturbation_baseline_df.columns if 'INPC' in col or 'PFC' in col]

percent_perturbation_baseline_sat_df = percent_perturbation_baseline_df.drop(columns=cols_to_drop)
max_percent_perturbation_baseline = percent_perturbation_baseline_sat_df.max(axis=1)
max_percent_perturbation_baseline_selected = max_percent_perturbation_baseline[selected_baseline_idx]
print(f'max percent perturbation for baseline solution {selected_baseline_idx}: {max_percent_perturbation_baseline_selected}')

percent_perturbation_IU_df = pd.read_csv(f"percent_degradation_max_allsols_IU_p90.csv", header=0)
max_percent_perturbation_IU_sat_df = percent_perturbation_IU_df.drop(columns=cols_to_drop)
max_percent_perturbation_IU = max_percent_perturbation_IU_sat_df.max(axis=1)

# filter out high-cooperation solutions that have higher perturbation than the baseline solution
high_coop_sols_IU_filtered = [sol for sol in high_coop_sols_IU if max_percent_perturbation_IU[sol] <= max_percent_perturbation_baseline_selected]
print(f'High-cooperation solutions with lower perturbation than the baseline\n: {np.array(high_coop_sols_IU_filtered).size}')

max percent perturbation for baseline solution 29: 0.748201552668758
High-cooperation solutions with lower perturbation than the baseline
: 416


In [5]:
# filter out robustness in IU df to only include solutions that have higher robustness than min robustness of baseline solutions
robustness_baseline_df = pd.read_csv(f'robustness_sim_p90_refset_base.csv', header=0)
min_robustness_baseline = robustness_baseline_df.iloc[29,:].min()  # get the minimum robustness across all objectives for the baseline solution
robustness_df_noReg_highrobustness = robustness_df_noReg[(robustness_df_noReg > min_robustness_baseline).all(axis=1)]
robustness_df_noReg_highrobustness_idx = robustness_df_noReg_highrobustness.index
solutions_highrobust_lowperturb = [sol for sol in high_coop_sols_IU_filtered if sol in robustness_df_noReg_highrobustness_idx]


## Identify Social Planner Solution

In [6]:
# find the maximum robustness values and index for each utility 
max_robutness_vals_sp = robustness_df_norm_noReg.max()
if max_robutness_vals_sp.shape != (3,):
    raise ValueError('Max robustness values shape is incorrect.')

diff_from_max = (max_robutness_vals_sp - robustness_df_norm_noReg)**2

sum_across_utils = diff_from_max.sum(axis=1)

# remove the 40th solution from high_coop_sols_IU
#high_coop_sols_IU = [sol for sol in high_coop_sols_IU if sol != 40 and sol != 45]
sum_across_utils_highcoop = sum_across_utils[high_coop_sols_IU_filtered]
sp_idx = sum_across_utils.idxmin()
sp_idx_highcoop = sum_across_utils_highcoop.idxmin()
print(f'Social planner solution is solution #{sp_idx} with robustness values:\n{robustness_df_noReg.loc[sp_idx]}')
print(f'Social planner solution is solution #{sp_idx_highcoop} with robustness values:\n{robustness_df_noReg.loc[sp_idx_highcoop]}')

Social planner solution is solution #552 with robustness values:
Watertown    0.66
Dryville     0.77
Fallsland    0.61
Name: 552, dtype: float64
Social planner solution is solution #552 with robustness values:
Watertown    0.66
Dryville     0.77
Fallsland    0.61
Name: 552, dtype: float64


### Identify Power Index Solution using the Power Index method

In [ ]:
max_robutness_vals_pg = robustness_df_norm_noReg.max()
alpha_numerator = max_robutness_vals_pg - robustness_df_norm_noReg
alpha_denominator = alpha_numerator.sum(axis=0) 
'''
if alpha_denominator.shape != (num_sol_dict[action_mode],):
    raise ValueError('Alpha denominator shape is incorrect.')
'''
alpha_vals = alpha_numerator.divide(alpha_denominator, axis=1)

if alpha_vals.shape != (num_sol_dict[action_mode], 3):
    raise ValueError('Alpha values shape is incorrect.')

print(f'Sums across alphas for each utility:\n{alpha_vals.sum(axis=0)}')

alpha_mean = alpha_vals.mean(axis=1)
alpha_std = alpha_vals.std(axis=1)               

CV_vals = alpha_std / alpha_mean

CV_vals_highcoop = CV_vals[high_coop_sols_IU_filtered]
# solution with lowest CV
cv_idx = CV_vals.idxmin()
cv_idx_highcoop = CV_vals_highcoop.idxmin()

print(f'Power Index solution is solution #{cv_idx} with robustness values:\n{robustness_df_noReg.loc[cv_idx]}')
print(f'Highly cooperative Power Index solution is solution #{cv_idx_highcoop} with robustness values:\n{robustness_df_noReg.loc[cv_idx_highcoop]}')

Sums across alphas for each utility:
Watertown    1.0
Dryville     1.0
Fallsland    1.0
dtype: float64
Power Index solution is solution #260 with robustness values:
Watertown    0.645
Dryville     0.645
Fallsland    0.575
Name: 260, dtype: float64
Highly cooperative Power Index solution is solution #260 with robustness values:
Watertown    0.645
Dryville     0.645
Fallsland    0.575
Name: 260, dtype: float64
Solution 363 has robustness values:
Watertown    0.585
Dryville     0.620
Fallsland    0.555
Name: 61, dtype: float64


In [8]:
# get the top 10 solutions with lowest CV
top_10_cv_indices = CV_vals_highcoop.nsmallest(10).index
top_10_cv_indices

Index([260, 530, 199, 53, 196, 151, 247, 616, 198, 33], dtype='int64')

### Robustness difference for the baseline reference set between p90 and avg conditions 

In [ ]:
# get robustness difference for the baseline reference set between p90 and avg conditions 
directory_avg = f'output_sim_avg_refset_base_reeval'
directory_p90 = f'output_sim_p90_refset_base_reeval'

robustness_base_avg = pd.read_csv(f'robustness_sim_avg_refset_base.csv', header=0)
robustness_base_p90 = pd.read_csv(f'robustness_sim_p90_refset_base.csv', header=0)

robustness_diff = robustness_base_avg - robustness_base_p90
robustness_diff_mean = robustness_diff.mean(axis=1)
robustness_diff_max = robustness_diff.max(axis=1)


# get the top 5 robustness diff mean values 
n = 5
top5_mean = np.argpartition(robustness_diff_mean.values, -n)[-n:]
top5_max = np.argpartition(robustness_diff_max.values, -n)[-n:]

print(f'Top 5 solutions with biggest mean diff in robustness: {top5_mean}')
print(f'Top 5 solutions with biggest max diff in robustness: {top5_max}')